# AI-Augmented Transdiagnostic Pleiotropy & Discordance Explorer for PGC Psychiatric GWAS

**Data attribution notice:** This notebook analyzes CC-BY 4.0 data from the OpenMed_AI PGC collection on Hugging Face, derived from Psychiatric Genomics Consortium (PGC) summary statistics. You must cite OpenMed_AI, PGC, and the original PGC publication/config used for each trait.

## Notebook goals

- Validate the currently exposed Hugging Face configs for six core psychiatric traits.
- Extract genome-wide-significant loci plus a top-hit panel per trait without loading the full billion-row collection into memory.
- Harmonize effect sizes and alleles across traits.
- Rank pleiotropic and discordant loci.
- Produce publication-ready figures, CSV tables, and an AI-ready interpretation prompt.

## Runtime expectation

End-to-end runtime is designed to stay under 60 minutes on a free Colab session or a standard laptop, assuming a stable internet connection to Hugging Face and UCSC.

In [ ]:
from __future__ import annotations

import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.decomposition import NMF, PCA
from upsetplot import UpSet, from_indicators

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from harmonize import (
    ATTRIBUTION_NOTICE,
    GENOME_WIDE_SIGNIFICANCE,
    annotate_with_nearest_gene,
    build_cross_trait_tables,
    default_trait_specs,
    download_refgene,
    extract_all_traits,
    summarize_trait_hits,
    trait_table,
)

RANDOM_STATE = 42
TOP_N = 15_000
P_THRESHOLD = GENOME_WIDE_SIGNIFICANCE
BACKEND = 'polars'
ROW_CAP = None

RESULTS_DIR = PROJECT_ROOT / 'results'
TABLE_DIR = RESULTS_DIR / 'tables'
FIG_DIR = RESULTS_DIR / 'figures'
CACHE_DIR = RESULTS_DIR / 'cache'
for path in [RESULTS_DIR, TABLE_DIR, FIG_DIR, CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 100)

print(ATTRIBUTION_NOTICE)
print(f'Project root: {PROJECT_ROOT}')

## 1. Inspect live configs before extraction

The notebook does not trust stale config names. It checks the OpenMed_AI dataset cards exposed through the `datasets` library at runtime and then resolves one usable config per trait. This matters for live collections because newly added configs can be malformed or partially converted.

As validated on **April 7, 2026**:

- `scz2022`, `bip2024`, `mdd2025`, and `adhd2022` are usable current defaults.
- `asd2019` is the latest Autism config currently exposed, even though some external planning documents may reference `asd2023`.
- `ptsd2024` exists, but its current parquet preview resolves to VCF-header metadata rather than harmonized summary-stat rows, so the runtime default falls back to `ptsd2019`.

In [ ]:
traits = default_trait_specs()
config_status = trait_table(traits)
display(config_status)

## 2. Extract reduced hit panels per trait

Each trait is reduced to:

- all loci passing genome-wide significance (`P < 5e-8`)
- a top-hit panel of up to 15,000 loci by p-value

The preferred backend uses Polars to lazily scan Hugging Face parquet shards. If that path fails for a dataset, the code falls back to `datasets` streaming with a heap-based top-hit tracker.

In [ ]:
trait_hits = extract_all_traits(
    traits=traits,
    p_threshold=P_THRESHOLD,
    top_n=TOP_N,
    backend=BACKEND,
    row_cap=ROW_CAP,
)

trait_summary = summarize_trait_hits(trait_hits)
trait_summary.to_csv(TABLE_DIR / 'trait_hit_counts.csv', index=False)
display(trait_summary)

## 3. Build the cross-trait summary objects

This step harmonizes alleles across traits, computes signed Z-scores, counts disorder-level significance support per locus, and creates the core tables used for plots and exports. We then annotate the lead representation of each locus with a simple nearest-gene lookup from UCSC `refGene`.

In [ ]:
cross_trait = build_cross_trait_tables(trait_hits, p_threshold=P_THRESHOLD)

summary_df = cross_trait['summary_df'].copy()
long_df = cross_trait['long_df'].copy()
membership_df = cross_trait['membership_df'].copy().set_index('match_key')
overlap_matrix = cross_trait['overlap_matrix'].copy()
overlap_matrix = overlap_matrix.set_index(overlap_matrix.columns[0])
zscore_matrix = cross_trait['zscore_matrix'].copy().set_index('match_key')

genes = download_refgene(CACHE_DIR)
summary_annotated = annotate_with_nearest_gene(summary_df, genes)

pleiotropic_top50 = (
    summary_annotated[summary_annotated['pleiotropy_score'] >= 2]
    .sort_values(['pleiotropy_score', 'fisher_combined_p', 'best_p'], ascending=[False, True, True])
    .head(50)
    .reset_index(drop=True)
)
discordant_top20 = (
    summary_annotated[summary_annotated['discordant_flag']]
    .sort_values(['pleiotropy_score', 'fisher_combined_p', 'best_p'], ascending=[False, True, True])
    .head(20)
    .reset_index(drop=True)
)

summary_annotated.to_csv(TABLE_DIR / 'cross_trait_variant_summary.csv', index=False)
zscore_matrix.reset_index().to_csv(TABLE_DIR / 'zscore_matrix.csv', index=False)
pleiotropic_top50.to_csv(TABLE_DIR / 'pleiotropic_loci_top50.csv', index=False)
discordant_top20.to_csv(TABLE_DIR / 'discordant_loci_top20.csv', index=False)

display(summary_annotated.head(10))

### Plot caption: UpSet overlap map

This figure answers a simple question: how many significant loci are shared by each disorder combination? Large intersections indicate dense transdiagnostic architecture. Sparse, trait-specific bars suggest more disorder-specific burden or differences in power.

In [ ]:
upset_membership = membership_df.astype(bool)
upset_data = from_indicators(upset_membership.columns.tolist(), upset_membership)

fig = plt.figure(figsize=(13, 6))
upset = UpSet(upset_data, subset_size='count', sort_by='cardinality', show_counts=True)
upset.plot(fig=fig)
plt.suptitle('Genome-wide-significant overlap across six psychiatric disorders', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'upset_overlap.png', bbox_inches='tight')
plt.show()

### Plot caption: Shared-loci heatmap

The diagonal shows the number of significant loci retained per trait. Off-diagonal cells show how many significant loci are shared between disorder pairs after harmonized matching. Darker cells highlight the strongest cross-disorder sharing candidates for mechanistic follow-up.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(overlap_matrix, annot=True, fmt='.0f', cmap='Reds', linewidths=0.5)
plt.title('Shared genome-wide-significant loci across disorder pairs')
plt.xlabel('Trait')
plt.ylabel('Trait')
plt.tight_layout()
plt.savefig(FIG_DIR / 'shared_loci_heatmap.png', bbox_inches='tight')
plt.show()

### Plot caption: Multi-trait Manhattan-style overview

This is a sentinel-loci Manhattan-style view built from the reduced hit panels rather than the full GWAS background. It is meant for rapid cross-trait comparison, not for replacing full per-trait Manhattan plots. Clusters of colored high-signal points suggest chromosomal regions worth detailed locus-level follow-up.

In [ ]:
manhattan_df = (
    long_df.sort_values('p', ascending=True)
    .groupby('trait', as_index=False)
    .head(1_500)
    .copy()
)
manhattan_df = manhattan_df.dropna(subset=['chr', 'pos'])
manhattan_df['neglog10p'] = -np.log10(manhattan_df['p'].clip(lower=1e-300))

chromosome_lengths = manhattan_df.groupby('chr')['pos'].max().sort_index()
chromosome_offsets = {}
running_offset = 0
for chrom, max_pos in chromosome_lengths.items():
    chromosome_offsets[chrom] = running_offset
    running_offset += int(max_pos)

manhattan_df['plot_x'] = manhattan_df.apply(
    lambda row: int(row['pos']) + chromosome_offsets[int(row['chr'])], axis=1
)
chromosome_centers = {
    chrom: chromosome_offsets[chrom] + chromosome_lengths[chrom] / 2.0
    for chrom in chromosome_lengths.index
}

plt.figure(figsize=(15, 6))
sns.scatterplot(
    data=manhattan_df,
    x='plot_x',
    y='neglog10p',
    hue='trait',
    palette='tab10',
    s=24,
    alpha=0.75,
    linewidth=0,
)
plt.axhline(-np.log10(P_THRESHOLD), color='black', linestyle='--', linewidth=1)
plt.xticks(list(chromosome_centers.values()), [str(c) for c in chromosome_centers.keys()], rotation=0)
plt.xlabel('Chromosome')
plt.ylabel('-log10(P)')
plt.title('Sentinel multi-trait Manhattan-style plot across retained hit panels')
plt.legend(title='Trait', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIG_DIR / 'multi_trait_manhattan.png', bbox_inches='tight')
plt.show()

### Plot caption: Disorder overlap network

Disorders are nodes and shared significant loci are weighted edges. Thick edges indicate strong shared-locus burden. This network offers a compact visual summary of transdiagnostic architecture across psychosis, mood, neurodevelopmental, and trauma-linked phenotypes.

In [ ]:
graph = nx.Graph()
for trait in overlap_matrix.index:
    graph.add_node(trait)

for row_trait in overlap_matrix.index:
    for col_trait in overlap_matrix.columns:
        if row_trait >= col_trait:
            continue
        weight = int(overlap_matrix.loc[row_trait, col_trait])
        if weight > 0:
            graph.add_edge(row_trait, col_trait, weight=weight)

layout = nx.spring_layout(graph, seed=RANDOM_STATE, weight='weight')
edge_weights = [graph[u][v]['weight'] for u, v in graph.edges()]
edge_widths = [1 + 5 * (weight / max(edge_weights)) for weight in edge_weights] if edge_weights else []

plt.figure(figsize=(8, 8))
nx.draw_networkx(
    graph,
    pos=layout,
    with_labels=True,
    node_size=2600,
    node_color='#d9ead3',
    edge_color='#52796f',
    width=edge_widths,
    font_size=12,
)
nx.draw_networkx_edge_labels(
    graph,
    pos=layout,
    edge_labels={(u, v): d['weight'] for u, v, d in graph.edges(data=True)},
    font_size=10,
)
plt.title('Shared-locus network across six psychiatric disorders')
plt.axis('off')
plt.tight_layout()
plt.savefig(FIG_DIR / 'disorder_network.png', bbox_inches='tight')
plt.show()

### Plot caption: Signed Z-score correlation and latent factors

The correlation heatmap summarizes whether retained loci tend to move in the same effect-direction space across traits. PCA loadings expose broad linear axes of shared architecture. NMF on `abs(Z)` provides a non-negative alternative focused on shared magnitude rather than direction.

In [ ]:
correlation = zscore_matrix.corr(min_periods=25)

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap='vlag', center=0, linewidths=0.5, fmt='.2f')
plt.title('Correlation of signed Z-scores across disorders')
plt.tight_layout()
plt.savefig(FIG_DIR / 'zscore_correlation_heatmap.png', bbox_inches='tight')
plt.show()

latent_input = zscore_matrix.fillna(0.0)
max_components = min(3, latent_input.shape[1], max(1, latent_input.shape[0] - 1))

if max_components >= 1:
    pca = PCA(n_components=max_components, random_state=RANDOM_STATE)
    pca.fit(latent_input)
    pca_loadings = pd.DataFrame(
        pca.components_.T,
        index=latent_input.columns,
        columns=[f'PC{i + 1}' for i in range(max_components)],
    )
    pca_loadings.to_csv(TABLE_DIR / 'pca_trait_loadings.csv')
    display(Markdown('**PCA trait loadings**'))
    display(pca_loadings)

    nmf = NMF(n_components=max_components, init='nndsvda', random_state=RANDOM_STATE, max_iter=1_000)
    nmf.fit(np.abs(latent_input) + 1e-6)
    nmf_loadings = pd.DataFrame(
        nmf.components_.T,
        index=latent_input.columns,
        columns=[f'NMF{i + 1}' for i in range(max_components)],
    )
    nmf_loadings.to_csv(TABLE_DIR / 'nmf_trait_loadings.csv')
    display(Markdown('**NMF trait loadings on abs(Z)**'))
    display(nmf_loadings)
else:
    print('Not enough loci were retained to compute latent factors.')

## 4. Prioritized locus tables

The pleiotropic table emphasizes cross-disorder support. The discordant table emphasizes sign flips after allele alignment. Both are suitable starting points for pathway review, literature synthesis, and follow-up fine-mapping.

In [ ]:
priority_columns = [
    'snp',
    'chr',
    'pos',
    'nearest_gene',
    'nearest_gene_distance_bp',
    'pleiotropy_score',
    'significant_traits',
    'discordant_flag',
    'discordant_pairs',
    'best_p',
    'fisher_combined_p',
]

display(Markdown('**Top pleiotropic loci**'))
display(pleiotropic_top50[priority_columns].head(20))

display(Markdown('**Top discordant loci**'))
display(discordant_top20[priority_columns].head(20))

## 5. AI interpretation layer

The goal is not to let an LLM free-associate over genetics. Instead, we export a constrained prompt that forces the model to work from the exact top loci tables, discuss plausible pathways, propose validation experiments, and separate well-supported interpretations from speculation.

In [ ]:
selected_configs = config_status[["trait", "selected_config", "selection_reason"]].copy()

report_text = textwrap.dedent(
    f"""
    # PGC Transdiagnostic Pleiotropy & Discordance Explorer Report

    Data attribution notice: {ATTRIBUTION_NOTICE}

    ## Runtime settings
    - Backend: {BACKEND}
    - Top hits retained per trait: {TOP_N:,}
    - Genome-wide-significance threshold: {P_THRESHOLD:.1e}
    - Approximate streaming row cap: {ROW_CAP}

    ## Selected dataset configs
    {selected_configs.to_string(index=False)}

    ## Trait extraction summary
    {trait_summary.to_string(index=False)}

    ## Top pleiotropic loci
    {pleiotropic_top50[priority_columns].head(20).to_string(index=False)}

    ## Top discordant loci
    {discordant_top20[priority_columns].head(20).to_string(index=False)}
    """
).strip() + "\n"

prompt_text = textwrap.dedent(
    f"""
    # LLM Interpretation Prompt

    Data attribution notice: {ATTRIBUTION_NOTICE}

    You are interpreting cross-disorder psychiatric GWAS summary statistics from the OpenMed_AI PGC collection.

    Input files:
    - {TABLE_DIR / 'pleiotropic_loci_top50.csv'}
    - {TABLE_DIR / 'discordant_loci_top20.csv'}
    - {TABLE_DIR / 'cross_trait_variant_summary.csv'}

    Tasks:
    1. Summarize the strongest transdiagnostic biological themes supported by the pleiotropic loci.
    2. Highlight any discordant loci that may indicate trait divergence, antagonistic pleiotropy, or phenotype heterogeneity.
    3. For the top genes, propose plausible pathways, tissues, and cell types to prioritize.
    4. Suggest drug-repurposing ideas, but clearly separate evidence-backed targets from speculative ideas.
    5. End with a table of high-priority validation experiments.

    Guardrails:
    - Work only from the supplied loci and genes.
    - Do not claim causality from summary statistics alone.
    - Flag uncertainty explicitly.
    - Distinguish literature-backed statements from hypotheses.
    - Keep OpenMed_AI, PGC, and the original PGC studies cited in the output.
    """
).strip() + "\n"

(RESULTS_DIR / 'analysis_report.md').write_text(report_text)
(RESULTS_DIR / 'llm_interpretation_prompt.md').write_text(prompt_text)

print(f"Wrote report to: {RESULTS_DIR / 'analysis_report.md'}")
print(f"Wrote LLM prompt to: {RESULTS_DIR / 'llm_interpretation_prompt.md'}")


## Future extensions

- Replace the temporary PTSD fallback with `ptsd2024` when the upstream parquet conversion is fixed.
- Add ancestry-stratified runs where metadata support that distinction cleanly.
- Add LD clumping and locus collapsing.
- Add eQTL, pQTL, and chromatin contact follow-up.
- Add pathway enrichment, drug-target overlays, and interactive dashboards.

This notebook is meant to be the first reproducible pass, not the last word.